# Experimento 02: Random Forest (Bagging y No Linealidad)

En este notebook escalamos la complejidad algorítmica hacia los modelos de **ensamblado basados en árboles (Bagging)**.

A diferencia de la Regresión Logística (que busca separar el texto con planos geométricos rectos), **Random Forest** construye múltiples árboles de decisión independientes que votan entre sí. Esto le permite encontrar relaciones complejas y no lineales en el lenguaje (por ejemplo, que la palabra "luz" signifique avería si va acompañada de "roja", pero signifique facturación si va acompañada de "recibo").

### Paso 1: Carga Express de Matrices

In [1]:
import scipy.sparse
import pandas as pd
import time

print("Iniciando carga de matrices en memoria (Modo MLOps)...")
start_load = time.time()

# 1. CARGA INGLÉS
en_X_train = scipy.sparse.load_npz("../data/features/en_X_train_tfidf.npz")
en_X_val = scipy.sparse.load_npz("../data/features/en_X_val_tfidf.npz")
en_X_test = scipy.sparse.load_npz("../data/features/en_X_test_tfidf.npz")

en_y_train = pd.read_csv("../data/features/en_y_train.csv").iloc[:, 0]
en_y_val = pd.read_csv("../data/features/en_y_val.csv").iloc[:, 0]
en_y_test = pd.read_csv("../data/features/en_y_test.csv").iloc[:, 0]

# 2. CARGA ESPAÑOL
es_X_train = scipy.sparse.load_npz("../data/features/es_X_train_tfidf.npz")
es_X_val = scipy.sparse.load_npz("../data/features/es_X_val_tfidf.npz")
es_X_test = scipy.sparse.load_npz("../data/features/es_X_test_tfidf.npz")

es_y_train = pd.read_csv("../data/features/es_y_train.csv").iloc[:, 0]
es_y_val = pd.read_csv("../data/features/es_y_val.csv").iloc[:, 0]
es_y_test = pd.read_csv("../data/features/es_y_test.csv").iloc[:, 0]

print(f"✅ Todos los datasets cargados en {round(time.time() - start_load, 2)} segundos.")

Iniciando carga de matrices en memoria (Modo MLOps)...
✅ Todos los datasets cargados en 0.34 segundos.


### Paso 2: Evaluación Control (Random Forest Sin SMOTE)

Establecemos el primer punto de control del Bosque Aleatorio. Vamos a evaluar si la arquitectura predictiva de árboles es capaz de resistir el desbalanceo masivo de clases por sí misma. 

La hipótesis inicial es que, aunque Random Forest encuentre patrones más complejos que la Regresión Logística, seguirá viéndose abrumado por el volumen de la clase mayoritaria (*Technical Support*) y ahogará a las minorías (*Service Outages*).

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
import time

# Función Fábrica para Random Forest
def train_evaluate_rf(X_train, y_train, X_val, y_val, exp_name):
    print(f"\n=======================================================")
    print(f"--- ENTRENANDO RANDOM FOREST - {exp_name} ---")
    print("⏳ Por favor, espera. Los árboles están creciendo (puede tardar más de 1 minuto)...")
    
    # 1. INSTANCIACIÓN DEL MODELO
    # n_estimators=100 (Crea 100 árboles de decisión)
    # n_jobs=-1 (VITAL: Obliga a usar todos los núcleos del procesador)
    # random_state=42 (Para asegurar reproducibilidad científica)
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    
    # 2. ENTRENAMIENTO
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = round(time.time() - start_train, 4)
    
    # 3. INFERENCIA
    start_inf = time.time()
    y_pred = model.predict(X_val)
    inf_time_ms = round((time.time() - start_inf) * 1000, 2)
    
    # 4. REPORTE COMPLETO
    print(f"✅ ¡Entrenamiento Finalizado!")
    print(f"[T. Entrenamiento: {train_time} seg | T. Inferencia: {inf_time_ms} ms]\n")
    print(classification_report(y_val, y_pred, zero_division=0))
    
    # 5. RETORNO DE VARIABLES (Para el Tracker central)
    f1_macro = round(f1_score(y_val, y_pred, average='macro', zero_division=0), 4)
    report_dict = classification_report(y_val, y_pred, output_dict=True, zero_division=0)
    f1_minority = round(report_dict.get('Service Outages and Maintenance', {}).get('f1-score', 0), 4)
    
    return train_time, inf_time_ms, f1_macro, f1_minority

# EJECUTAMOS CONTROL (SIN SMOTE)
en_train_time_rf_none, en_inf_time_rf_none, en_f1_rf_none, en_f1_min_rf_none = train_evaluate_rf(
    en_X_train, en_y_train, en_X_val, en_y_val, "INGLÉS (SIN SMOTE)"
)

es_train_time_rf_none, es_inf_time_rf_none, es_f1_rf_none, es_f1_min_rf_none = train_evaluate_rf(
    es_X_train, es_y_train, es_X_val, es_y_val, "ESPAÑOL (SIN SMOTE)"
)


--- ENTRENANDO RANDOM FOREST - INGLÉS (SIN SMOTE) ---
⏳ Por favor, espera. Los árboles están creciendo (puede tardar más de 1 minuto)...
✅ ¡Entrenamiento Finalizado!
[T. Entrenamiento: 4.9768 seg | T. Inferencia: 53.2 ms]

                                 precision    recall  f1-score   support

           Billing and Payments       0.94      0.76      0.84       381
               Customer Service       0.59      0.55      0.57       569
                     IT Support       0.93      0.34      0.50       451
                Product Support       0.68      0.50      0.58       701
            Sales and Pre-Sales       1.00      0.28      0.44       115
Service Outages and Maintenance       0.92      0.55      0.69       149
              Technical Support       0.54      0.90      0.68      1102

                       accuracy                           0.64      3468
                      macro avg       0.80      0.55      0.61      3468
                   weighted avg       0.70  

### Paso 3: Estudio de Ablación (Inyección Sintética con SMOTE)

El modelo Random Forest sin balancear ha demostrado ser un clasificador "conservador": maximiza la Precisión pero sufre de un bajo Recall en las clases minoritarias (*Service Outages*). 

Para forzar a los árboles de decisión a aprender las reglas geométricas de las averías, aplicamos **SMOTE**. Al igualar el volumen de todas las clases en la matriz de entrenamiento, eliminamos el sesgo probabilístico. Esperamos observar una caída natural en la Precisión global (el modelo arriesgará más) a cambio de un aumento crítico en el Recall (menos escapes).

In [3]:
from imblearn.over_sampling import SMOTE
import time

print("--- INICIANDO INYECCIÓN SINTÉTICA (SMOTE) ---")

# 1. INSTANCIACIÓN DE SMOTE
smote = SMOTE(random_state=42)

# 2. TRANSFORMACIÓN INGLÉS
print("\nAplicando SMOTE al dataset en Inglés...")
start_smote_en = time.time()
en_X_train_smote, en_y_train_smote = smote.fit_resample(en_X_train, en_y_train)
print(f"✅ Inglés balanceado en {round(time.time() - start_smote_en, 2)} seg.")

# 3. TRANSFORMACIÓN ESPAÑOL
print("Aplicando SMOTE al dataset en Español...")
start_smote_es = time.time()
es_X_train_smote, es_y_train_smote = smote.fit_resample(es_X_train, es_y_train)
print(f"✅ Español balanceado en {round(time.time() - start_smote_es, 2)} seg.")

# 4. RE-ENTRENAMIENTO (Test de Fuego)
# Usamos las nuevas matrices "dopadas" que ahora son mucho más masivas
# OJO: Al haber más datos, el Random Forest tardará bastante más en entrenar.
en_train_time_rf_smote, en_inf_time_rf_smote, en_f1_rf_smote, en_f1_min_rf_smote = train_evaluate_rf(
    en_X_train_smote, en_y_train_smote, en_X_val, en_y_val, "INGLÉS (CON SMOTE)"
)

es_train_time_rf_smote, es_inf_time_rf_smote, es_f1_rf_smote, es_f1_min_rf_smote = train_evaluate_rf(
    es_X_train_smote, es_y_train_smote, es_X_val, es_y_val, "ESPAÑOL (CON SMOTE)"
)

--- INICIANDO INYECCIÓN SINTÉTICA (SMOTE) ---

Aplicando SMOTE al dataset en Inglés...
✅ Inglés balanceado en 0.65 seg.
Aplicando SMOTE al dataset en Español...
✅ Español balanceado en 0.59 seg.

--- ENTRENANDO RANDOM FOREST - INGLÉS (CON SMOTE) ---
⏳ Por favor, espera. Los árboles están creciendo (puede tardar más de 1 minuto)...
✅ ¡Entrenamiento Finalizado!
[T. Entrenamiento: 15.6008 seg | T. Inferencia: 65.79 ms]

                                 precision    recall  f1-score   support

           Billing and Payments       0.91      0.81      0.86       381
               Customer Service       0.59      0.64      0.61       569
                     IT Support       0.77      0.47      0.58       451
                Product Support       0.72      0.60      0.66       701
            Sales and Pre-Sales       0.95      0.50      0.65       115
Service Outages and Maintenance       0.78      0.68      0.73       149
              Technical Support       0.62      0.82      0.71     

### Paso 4: Consolidación MLOps (Registro Central de Experimentos)

Volcamos los 4 modelos de Random Forest (Inglés/Español, Sin/Con SMOTE) en los Trackers Maestros. 
Registramos explícitamente en la columna de hiperparámetros que el ensamblado se realizó con `n_estimators=100` (valor base). Esta métrica de F1-Macro (0.68) se convierte en el nuevo rival a batir para los algoritmos de Boosting en la siguiente fase.

In [4]:
import pandas as pd

print("--- GUARDANDO EXPERIMENTOS EN EL TRACKER CENTRAL ---")

# 1. CARGAMOS LOS TRACKERS FÍSICOS
tracker_en = pd.read_csv("../data/processed/tracker_en.csv")
tracker_es = pd.read_csv("../data/processed/tracker_es.csv")

# 2. EMPAQUETAMOS (INGLÉS)
nuevos_experimentos_en = [
    {
        'exp_id': 'EN_L1_TFIDF_RF_NONE',
        'target_level': 'queue',
        'vectorization': 'tfidf',
        'model': 'random_forest',
        'balancing': 'none',
        'hyperparameters': 'n_estimators=100',
        'train_time_sec': en_train_time_rf_none,
        'inference_time_ms': en_inf_time_rf_none,
        'f1_macro': en_f1_rf_none,
        'f1_minority_class': en_f1_min_rf_none
    },
    {
        'exp_id': 'EN_L1_TFIDF_RF_SMOTE',
        'target_level': 'queue',
        'vectorization': 'tfidf',
        'model': 'random_forest',
        'balancing': 'smote',
        'hyperparameters': 'n_estimators=100',
        'train_time_sec': en_train_time_rf_smote,
        'inference_time_ms': en_inf_time_rf_smote,
        'f1_macro': en_f1_rf_smote,
        'f1_minority_class': en_f1_min_rf_smote
    }
]

# 3. EMPAQUETAMOS (ESPAÑOL)
nuevos_experimentos_es = [
    {
        'exp_id': 'ES_L1_TFIDF_RF_NONE',
        'target_level': 'queue',
        'vectorization': 'tfidf',
        'model': 'random_forest',
        'balancing': 'none',
        'hyperparameters': 'n_estimators=100',
        'train_time_sec': es_train_time_rf_none,
        'inference_time_ms': es_inf_time_rf_none,
        'f1_macro': es_f1_rf_none,
        'f1_minority_class': es_f1_min_rf_none
    },
    {
        'exp_id': 'ES_L1_TFIDF_RF_SMOTE',
        'target_level': 'queue',
        'vectorization': 'tfidf',
        'model': 'random_forest',
        'balancing': 'smote',
        'hyperparameters': 'n_estimators=100',
        'train_time_sec': es_train_time_rf_smote,
        'inference_time_ms': es_inf_time_rf_smote,
        'f1_macro': es_f1_rf_smote,
        'f1_minority_class': es_f1_min_rf_smote
    }
]

# 4. INYECTAMOS Y SOBREESCRIBIMOS
tracker_en = pd.concat([tracker_en, pd.DataFrame(nuevos_experimentos_en)], ignore_index=True)
tracker_en.to_csv("../data/processed/tracker_en.csv", index=False)

tracker_es = pd.concat([tracker_es, pd.DataFrame(nuevos_experimentos_es)], ignore_index=True)
tracker_es.to_csv("../data/processed/tracker_es.csv", index=False)

print("✅ Todos los experimentos han sido registrados con éxito.")

# 5. AUDITORÍA VISUAL DEL PROGRESO
print("\n--- TRACKER MAESTRO ACTUALIZADO (INGLÉS) ---")
display(tracker_en)
print("\n--- TRACKER MAESTRO ACTUALIZADO (ESPAÑOL) ---")
display(tracker_es)

--- GUARDANDO EXPERIMENTOS EN EL TRACKER CENTRAL ---
✅ Todos los experimentos han sido registrados con éxito.

--- TRACKER MAESTRO ACTUALIZADO (INGLÉS) ---


,exp_id,target_level,vectorization,model,balancing,train_time_sec,inference_time_ms,f1_macro,f1_minority_class,hyperparameters
0,EN_L1_TFIDF_MNB_NONE,queue,tfidf,mnb,none,0.0475,2.61,0.3347,0.2775,baseline default
1,EN_L1_TFIDF_LOGREG_NONE,queue,tfidf,logreg,none,2.3864,2.48,0.4598,0.5614,max_iter=1000
2,EN_L1_TFIDF_LOGREG_SMOTE,queue,tfidf,logreg,smote,5.9526,1.73,0.5079,0.5605,max_iter=1000
3,EN_L1_TFIDF_RF_NONE,queue,tfidf,random_forest,none,4.9768,53.20,0.6128,0.6891,n_estimators=100
4,EN_L1_TFIDF_RF_SMOTE,queue,tfidf,random_forest,smote,15.6008,65.79,0.6849,0.7266,n_estimators=100



--- TRACKER MAESTRO ACTUALIZADO (ESPAÑOL) ---


,exp_id,target_level,vectorization,model,balancing,train_time_sec,inference_time_ms,f1_macro,f1_minority_class,hyperparameters
0,ES_L1_TFIDF_MNB_NONE,queue,tfidf,mnb,none,0.0425,2.42,0.3088,0.0870,baseline default
1,ES_L1_TFIDF_LOGREG_NONE,queue,tfidf,logreg,none,2.6192,1.92,0.4522,0.5066,max_iter=1000
2,ES_L1_TFIDF_LOGREG_SMOTE,queue,tfidf,logreg,smote,5.7504,2.17,0.5007,0.5938,max_iter=1000
3,ES_L1_TFIDF_RF_NONE,queue,tfidf,random_forest,none,4.8221,62.61,0.5816,0.6121,n_estimators=100
4,ES_L1_TFIDF_RF_SMOTE,queue,tfidf,random_forest,smote,15.7321,65.97,0.6502,0.7000,n_estimators=100
